In [2]:
%pip install -q mediapipe

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install mediapipe opencv-python

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# مسار الموديل
MODEL_PATH = "hand_landmarker.task"

# إعدادات الموديل
BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)

# فتح الكاميرا
cap = cv2.VideoCapture(0)

with HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # OpenCV → RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        # التوقيت مطلوب في وضع الفيديو
        timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        # رسم النقاط
        if result.hand_landmarks:
            for hand_landmarks in result.hand_landmarks:
                for lm in hand_landmarks:
                    h, w, _ = frame.shape
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

        cv2.imshow("Hand Landmarker", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()


In [6]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time

# ================== MODEL PATHS ==================
POSE_MODEL_PATH = "pose_landmarker_full.task"
HAND_MODEL_PATH = "hand_landmarker.task"

# ================== BASE OPTIONS ==================
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

# ================== POSE SETUP ==================
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1
)

# ================== HAND SETUP ==================
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)

# ================== CAMERA ==================
cap = cv2.VideoCapture(0)
start_time = time.time()

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape

        # Convert BGR → RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        # Proper timestamp for VIDEO mode
        timestamp_ms = int((time.time() - start_time) * 1000)

        # ================== POSE DETECTION ==================
        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        if pose_result.pose_landmarks:
            landmarks = pose_result.pose_landmarks[0]

            # Important upper body landmarks
            LEFT_SHOULDER = 11
            RIGHT_SHOULDER = 12
            LEFT_ELBOW = 13
            RIGHT_ELBOW = 14
            LEFT_WRIST = 15
            RIGHT_WRIST = 16
            LEFT_HIP = 23
            RIGHT_HIP = 24

            important_points = [
                LEFT_SHOULDER, RIGHT_SHOULDER,
                LEFT_ELBOW, RIGHT_ELBOW,
                LEFT_WRIST, RIGHT_WRIST,
                LEFT_HIP, RIGHT_HIP
            ]

            # Draw pose points
            for idx in important_points:
                lm = landmarks[idx]
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 6, (0, 255, 0), -1)

            # -------- Arm Raised Detection --------
            left_shoulder = landmarks[LEFT_SHOULDER]
            left_hip = landmarks[LEFT_HIP]

            if left_shoulder.y < left_hip.y - 0.05:
                cv2.putText(frame, "Arm Raised",
                            (30, 40),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, (0, 255, 0), 2)

            # -------- Shrug Detection --------
            if left_shoulder.y < 0.3:
                cv2.putText(frame, "Shrug Detected!",
                            (30, 80),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, (0, 0, 255), 2)

        # ================== HAND DETECTION ==================
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if hand_result.hand_landmarks:
            for hand_landmarks in hand_result.hand_landmarks:
                for lm in hand_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 5, (255, 0, 0), -1)

            cv2.putText(frame, "Hand Detected",
                        (30, 120),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1, (255, 0, 0), 2)

        # ================== DISPLAY ==================
        cv2.imshow("Pose + Hand Tracking", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
# this is the base code for only and obly hands

In [5]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
import math

MODEL_PATH = "pose_landmarker_full.task"

BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1
)

cap = cv2.VideoCapture(0)

with PoseLandmarker.create_from_options(options) as landmarker:

    start_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        timestamp_ms = int((time.time() - start_time) * 1000)

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks:

            landmarks = result.pose_landmarks[0]
            h, w, _ = frame.shape

            # Important upper body landmarks
            LEFT_SHOULDER = 11
            RIGHT_SHOULDER = 12
            LEFT_ELBOW = 13
            RIGHT_ELBOW = 14
            LEFT_WRIST = 15
            RIGHT_WRIST = 16
            LEFT_HIP = 23
            RIGHT_HIP = 24

            important_points = [
                LEFT_SHOULDER, RIGHT_SHOULDER,
                LEFT_ELBOW, RIGHT_ELBOW,
                LEFT_WRIST, RIGHT_WRIST,
                LEFT_HIP, RIGHT_HIP
            ]

            for idx in important_points:
                lm = landmarks[idx]
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 6, (0, 255, 0), -1)

            # -------- Detect Shoulder Shrug --------
            left_shoulder = landmarks[LEFT_SHOULDER]
            left_hip = landmarks[LEFT_HIP]

            shoulder_height = left_shoulder.y
            hip_height = left_hip.y

            if shoulder_height < hip_height - 0.05:
                cv2.putText(frame, "Arm Raised", (30, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Shrug detection (shoulder raised too much)
            if left_shoulder.y < 0.3:
                cv2.putText(frame, "Shrug Detected!", (30, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        cv2.imshow("Pose Tracking", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 